# 🔬 MetaDerm-XA: Training Pipeline (Colab T4 GPU)

This notebook trains all **4 model variants** for the MetaDerm-XA skin lesion classification project on a T4 GPU.

**Models:**
1. Baseline (Vision-Only)
2. Concatenation Fusion
3. Cross-Attention Fusion (MetaDerm-XA core)
4. Quantum-Inspired Fusion

**Estimated total training time: ~6-8 hours on T4 GPU**

---

## Prerequisites

Before running this notebook:
1. Upload your **entire `skin-lesion-project` folder** to Google Drive
2. Make sure the folder path on Drive is: `My Drive/skin-lesion-project/`
3. Set Runtime → Change runtime type → **T4 GPU**

## Step 1: Verify GPU & Mount Google Drive

In [ ]:
# Check GPU
!nvidia-smi
print()

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Copy Project to Colab Local Storage

Copying from Drive to Colab's local SSD gives **much faster I/O** during training.

> ⚠️ If your folder is at a different path on Drive, update the `DRIVE_PROJECT_PATH` below.

In [ ]:
import os
import shutil

# === UPDATE THIS PATH IF YOUR FOLDER IS NAMED DIFFERENTLY ===
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/skin-lesion-project'
LOCAL_PROJECT_PATH = '/content/skin-lesion-project'

# Verify the project exists on Drive
if not os.path.exists(DRIVE_PROJECT_PATH):
    print(f"❌ Project not found at: {DRIVE_PROJECT_PATH}")
    print("Please upload your skin-lesion-project folder to Google Drive.")
    print("\nAvailable folders in My Drive:")
    for item in os.listdir('/content/drive/MyDrive/'):
        print(f"  📁 {item}")
else:
    print(f"✅ Found project at: {DRIVE_PROJECT_PATH}")
    
    # Copy to local storage (much faster I/O)
    if os.path.exists(LOCAL_PROJECT_PATH):
        shutil.rmtree(LOCAL_PROJECT_PATH)
    
    print("📦 Copying project to Colab local storage (this may take a few minutes for 33K images)...")
    shutil.copytree(DRIVE_PROJECT_PATH, LOCAL_PROJECT_PATH)
    print("✅ Copy complete!")
    
    # Quick verification
    img_dir = os.path.join(LOCAL_PROJECT_PATH, 'data/raw/isic2020/images')
    if os.path.exists(img_dir):
        jpg_count = len([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
        print(f"📊 Found {jpg_count} JPEG images")
    
    csv_path = os.path.join(LOCAL_PROJECT_PATH, 'data/raw/isic2020/train.csv')
    if os.path.exists(csv_path):
        import pandas as pd
        df = pd.read_csv(csv_path)
        print(f"📊 CSV has {len(df)} rows")
        print(f"📊 Malignant: {df['target'].sum()} ({df['target'].mean()*100:.1f}%)")

## Step 3: Install Dependencies

In [ ]:
%cd /content/skin-lesion-project

# Install project dependencies
!pip install -q timm albumentations tensorboard tqdm scikit-learn

# Verify imports
import timm
import albumentations
print(f"\n✅ timm: {timm.__version__}")
print(f"✅ albumentations: {albumentations.__version__}")
print(f"✅ torch: {torch.__version__}")
print(f"✅ CUDA: {torch.version.cuda}")

## Step 4: Verify Data & Preprocessing

Check that the data splits are ready. If not, run preprocessing.

In [ ]:
import os

processed_dir = 'data/processed'
splits_exist = all(
    os.path.exists(os.path.join(processed_dir, f'{split}.csv'))
    for split in ['train', 'val', 'test']
)

if splits_exist:
    print("✅ Data splits already exist. Skipping preprocessing.")
    import pandas as pd
    for split in ['train', 'val', 'test']:
        df = pd.read_csv(os.path.join(processed_dir, f'{split}.csv'))
        mal = df['target'].sum()
        print(f"   {split}: {len(df)} images, {int(mal)} malignant ({mal/len(df)*100:.1f}%)")
else:
    print("⚙️ Running preprocessing...")
    !python src/data/preprocess.py
    print("✅ Preprocessing complete!")

---

## 🏋️ Model Training

Each cell trains one model variant. You can see **live progress bars** with loss and AUROC.

| Model | Batch Size | Est. Time (T4) |
|-------|-----------|----------------|
| Baseline (vision-only) | 48 | ~1.5 hrs |
| Concat Fusion | 48 | ~1.5 hrs |
| Cross-Attention | 32 | ~2 hrs |
| Quantum-Inspired | 32 | ~2 hrs |

In [ ]:
# ============================================================
# MODEL 1/4: BASELINE (Vision-Only)
# ============================================================
print("=" * 60)
print("  [1/4] Training BASELINE model (vision-only)")
print("=" * 60)

!python src/train.py --config configs/colab/baseline.yaml

print("\n✅ Baseline training complete!")

In [ ]:
# ============================================================
# MODEL 2/4: CONCATENATION FUSION
# ============================================================
print("=" * 60)
print("  [2/4] Training CONCAT FUSION model")
print("=" * 60)

!python src/train.py --config configs/colab/concat_fusion.yaml

print("\n✅ Concat Fusion training complete!")

In [ ]:
# ============================================================
# MODEL 3/4: CROSS-ATTENTION (MetaDerm-XA Core)
# ============================================================
print("=" * 60)
print("  [3/4] Training CROSS-ATTENTION model (MetaDerm-XA core)")
print("=" * 60)

!python src/train.py --config configs/colab/cross_attention.yaml

print("\n✅ Cross-Attention training complete!")

In [ ]:
# ============================================================
# MODEL 4/4: QUANTUM-INSPIRED FUSION
# ============================================================
print("=" * 60)
print("  [4/4] Training QUANTUM-INSPIRED FUSION model")
print("=" * 60)

!python src/train.py --config configs/colab/quantum_fusion.yaml

print("\n✅ Quantum Fusion training complete!")

---

## 📊 View Training Results

In [ ]:
import json
import os

results_dir = 'reports/results'
models = ['baseline', 'concat_fusion', 'cross_attention', 'quantum_fusion']

print("=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)
print(f"{'Model':<22} {'Best Val AUROC':<16} {'Epochs':<10} {'Params'}")
print("-" * 60)

for model in models:
    summary_path = os.path.join(results_dir, f'{model}_train_summary.json')
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            s = json.load(f)
        print(f"{s['model_name']:<22} {s['best_val_auroc']:<16.4f} {s['epochs_trained']:<10} {s['n_params']:,}")
    else:
        print(f"{model:<22} NOT TRAINED")

print("=" * 60)

In [ ]:
# Display training curves
from IPython.display import display, Image as IPImage
import os

results_dir = 'reports/results'
models = ['baseline', 'concat_fusion', 'cross_attention', 'quantum_fusion']

for model in models:
    curve_path = os.path.join(results_dir, f'{model}_training_curves.png')
    if os.path.exists(curve_path):
        print(f"\n{'='*40}")
        print(f"  {model}")
        print(f"{'='*40}")
        display(IPImage(filename=curve_path))

---

## 💾 Save Results Back to Google Drive

Copy all checkpoints, training curves, and summaries back to Drive so you don't lose them when the Colab session expires.

In [ ]:
import shutil
import os

DRIVE_PROJECT_PATH = '/content/drive/MyDrive/skin-lesion-project'

# Copy checkpoints
src_ckpt = '/content/skin-lesion-project/checkpoints'
dst_ckpt = os.path.join(DRIVE_PROJECT_PATH, 'checkpoints')
if os.path.exists(src_ckpt):
    os.makedirs(dst_ckpt, exist_ok=True)
    for f in os.listdir(src_ckpt):
        shutil.copy2(os.path.join(src_ckpt, f), os.path.join(dst_ckpt, f))
    print(f"✅ Checkpoints saved to Drive ({len(os.listdir(src_ckpt))} files)")

# Copy reports/results
src_reports = '/content/skin-lesion-project/reports/results'
dst_reports = os.path.join(DRIVE_PROJECT_PATH, 'reports', 'results')
if os.path.exists(src_reports):
    os.makedirs(dst_reports, exist_ok=True)
    for f in os.listdir(src_reports):
        shutil.copy2(os.path.join(src_reports, f), os.path.join(dst_reports, f))
    print(f"✅ Training results saved to Drive ({len(os.listdir(src_reports))} files)")

# Copy TensorBoard runs
src_runs = '/content/skin-lesion-project/runs'
dst_runs = os.path.join(DRIVE_PROJECT_PATH, 'runs')
if os.path.exists(src_runs):
    if os.path.exists(dst_runs):
        shutil.rmtree(dst_runs)
    shutil.copytree(src_runs, dst_runs)
    print(f"✅ TensorBoard logs saved to Drive")

print("\n🎉 All results saved to Google Drive! Your Colab session can now safely expire.")

---

## ✅ Done!

All 4 models are trained and saved. Next steps:

1. **Download checkpoints** from Google Drive to your local `skin-lesion-project/checkpoints/`
2. **Run evaluation**: `python src/evaluate.py --config configs/colab/cross_attention.yaml`
3. **Launch Streamlit demo**: `streamlit run app/streamlit_app.py`
4. **Generate counterfactual analysis** for your report